# Keras Functional API の概要

**学習目標**
1. 柔軟なモデル構築のための Keras Functional API を理解する。
2. Wide & Deep モデル（記憶と汎化の統合）を実装する。
3. Keras 前処理レイヤ（`Discretization`、`HashedCrossing`、`Embedding`）を使用する。

## はじめに

前のノートブックでは、Keras Sequential API について学びました。[Keras Functional API](https://keras.io/guides/functional_api/) は、より柔軟なモデル構築の別のアプローチを提供します。Functional API を使用すると、より複雑なトポロジ、複数の入力レイヤや出力レイヤ、共有レイヤ、または非順次データフロー（残差接続レイヤなど）を持つモデルを構築できます。

このノートブックでは、Functional API と前処理レイヤを使用して Wide & Deep モデルを構築します。

### Wide & Deep モデル
Wide & Deep モデルの背景にある考え方は、wide（線形）モデルと deep（ディープラーニング）モデルを組み合わせて、記憶（memorization）と汎化（generalization）という 2 つの学習手法を両立させることです。元の研究論文はこちらで確認できます: [Wide & Deep Learning for Recommender Systems](https://arxiv.org/abs/1606.07792)。

<img src='assets/wide_deep.png' width='80%'>
<sup>(画像出典: https://ai.googleblog.com/2016/06/wide-deep-learning-better-together-with.html)</sup>

モデルの wide 部分は「記憶」の要素に関連付けられています。この場合、交差特徴量（crossed features）の大規模なセットを使用して線形モデルをトレーニングし、この関連データと割り当てられたラベルの相関関係を学習します。モデルの deep 部分は「汎化」の要素に関連付けられており、特徴量に埋め込みベクトル（embedding vectors）を使用します。最適な埋め込みはトレーニング プロセスを通じて学習されます。これら 2 つの手法はそれぞれ単独でもうまく機能しますが、Wide & Deep モデルはこれらの手法を組み合わせることで優れた性能を発揮します。

## セットアップ
このラボに必要なライブラリをインポートすることから始めます。

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import datetime
import shutil

import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import Model
from keras.callbacks import TensorBoard
from keras.layers import (
    CategoryEncoding,
    Concatenate,
    Dense,
    Discretization,
    Embedding,
    Flatten,
    HashedCrossing,
    Input,
)
from keras.metrics import RootMeanSquaredError
from matplotlib import pyplot as plt

In [ ]:
%matplotlib inline

### 生データの読み込み

この一連のノートブックの最初のノートブックで作成した CSV ファイルを使用して、taxifare データセットを使用します。これらのファイルは `../data` に保存されています。

In [ ]:
!ls -l ../data/*.csv

### tf.data を使用した CSV ファイルの読み込み

上記の CSV ファイルからデータを読み込むためのこれらの関数は、[前のノートブック](2_dataset_api.ipynb) で作成しました。

`tf.data` API は、効率的にデータを読み込み前処理します。
- `parse_csv`: CSV の行をパースして特徴量とラベルに変換します。複数の入力を受け取る Functional API との互換性のため、特徴量はタプルとして返されます。
- `create_dataset`: `parse_csv` のマッピング、リピート、シャッフル（トレーニング用）、バッチ処理などを含め、CSV ファイルから `tf.data.Dataset` を構築します。

In [ ]:
def parse_csv(row):
    ds = tf.strings.split(row, ",")
    # ラベル: fare_amount
    label = tf.strings.to_number(ds[0])
    # 特徴量: pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude
    feature = tf.strings.to_number(ds[2:6])  # 一部の特徴量のみを使用
    # 特徴量を個別に処理できるようにタプルで渡す
    return (feature[0], feature[1], feature[2], feature[3]), label


def create_dataset(pattern, batch_size, mode="eval"):
    ds = tf.data.TextLineDataset(pattern).skip(1)
    ds = ds.map(parse_csv).repeat()
    if mode == "train":
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(batch_size, drop_remainder=True)
    return ds

## Keras Functional API による Wide & Deep モデルの構築

2 つの部分からなる Wide & Deep モデルを構築します。
1. **Wide パス:** 特定の特徴量の相互作用の記憶用（多くの場合、カテゴリカル特徴量/交差特徴量を使用）。
2. **Deep パス:** パターンの汎化用（多くの場合、DNN を介した数値特徴量/埋め込みを使用）。
その後、予測のために出力を結合します。Functional API はこの複雑な構造を処理します。

### 入力レイヤの定義

Functional API を使用すると、各入力特徴量に固有の [`keras.Input` レイヤ](https://keras.io/api/layers/core_layers/input/) が割り当てられます。これらはデータのエントリ ポイントを定義します。
4 つの座標特徴量に対して `Input` オブジェクトを作成し、それぞれの `name`、`shape`（スカラー）、および `dtype` を指定します。

In [ ]:
INPUT_COLS = [
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
]

inputs = {
    colname: Input(name=colname, shape=(1,), dtype="float32")
    for colname in INPUT_COLS
}

### 前処理ロジックの定義

次に、特徴量の前処理を定義します。
1. **バケット化（Bucketization）:** `Discretization` レイヤは、事前定義されたバケット（`latbuckets`、`lonbuckets`）に割り当てることで、連続値の緯度/経度入力をカテゴリカル特徴量に変換します。
2. **特徴量の交差（Feature Crossing）:** `HashedCrossing` レイヤは、これらのバケット化された特徴量を組み合わせて、相互作用特徴量を作成します（例: 乗車場所、降車場所、乗車-降車の相互作用）。これはモデルの wide 部分に役立ちます。

Functional API を使用すると、各接続を個別に定義できるため、より複雑なモデル アーキテクチャが可能になります。

In [ ]:
dnn_hidden_units = [32, 8]
NBUCKETS = 16

# バケット化の境界を定義
latbuckets = np.linspace(start=40.5, stop=41.0, num=NBUCKETS).tolist()
lonbuckets = np.linspace(start=-74.2, stop=-73.7, num=NBUCKETS).tolist()

# Discretization レイヤによるバケット化
plon = Discretization(lonbuckets, name="plon_bkt")(inputs["pickup_longitude"])
plat = Discretization(latbuckets, name="plat_bkt")(inputs["pickup_latitude"])
dlon = Discretization(lonbuckets, name="dlon_bkt")(inputs["dropoff_longitude"])
dlat = Discretization(latbuckets, name="dlat_bkt")(inputs["dropoff_latitude"])

# HashedCrossing レイヤによる特徴量の交差
p_fc = HashedCrossing(num_bins=(NBUCKETS + 1) ** 2, name="p_fc")((plon, plat))
d_fc = HashedCrossing(num_bins=(NBUCKETS + 1) ** 2, name="d_fc")((dlon, dlat))
pd_fc = HashedCrossing(num_bins=(NBUCKETS + 1) ** 4, name="pd_fc")((p_fc, d_fc))

### Deep パスの構築

Deep パスは汎化を処理します。
1. `pd_fc` 交差特徴量は、`Embedding` レイヤを使用して埋め込まれ、密なベクトル表現が作成されます。
2. これらの埋め込みは、元の数値入力座標と連結（concatenate）されます。
3. 結果は、ReLU 活性化関数を持つ一連の `Dense` レイヤに渡されます。

In [ ]:
# Embedding レイヤによる埋め込み
pd_embed = Embedding(
    input_dim=(NBUCKETS + 1) ** 4, output_dim=10, name="pd_embed"
)(pd_fc)

# 連結してディープ ネットワークの入力を定義
deep = Concatenate(name="deep_input")(
    [
        inputs["pickup_longitude"],
        inputs["pickup_latitude"],
        inputs["dropoff_longitude"],
        inputs["dropoff_latitude"],
        Flatten(name="flatten_embedding")(pd_embed),
    ]
)

# 隠れ Dense レイヤを追加
for i, num_nodes in enumerate(dnn_hidden_units, start=1):
    deep = Dense(num_nodes, activation="relu", name=f"hidden_{i}")(deep)

### Wide パスの構築

Wide パスは記憶を処理します。
1. 交差特徴量（`p_fc`、`d_fc`、`pd_fc`）は、`CategoryEncoding` レイヤを使用して one-hot エンコードされます。
2. これらの one-hot エンコードされた特徴量は連結され、モデルの wide 部分の入力を形成します。

In [ ]:
# CategoryEncoding レイヤによる one-hot エンコーディング
p_onehot = CategoryEncoding(num_tokens=(NBUCKETS + 1) ** 2, name="p_onehot")(
    p_fc
)
d_onehot = CategoryEncoding(num_tokens=(NBUCKETS + 1) ** 2, name="d_onehot")(
    d_fc
)
pd_onehot = CategoryEncoding(num_tokens=(NBUCKETS + 1) ** 4, name="pd_onehot")(
    pd_fc
)

# 連結してワイド ネットワークの入力を定義
wide = Concatenate(name="wide_input")([p_onehot, d_onehot, pd_onehot])

### Wide パスと Deep パスの結合

`deep` パスと `wide` パスの出力が連結されます。この結合されたテンソルは、予測を生成するために 1 つのユニットを持つ最終的な `Dense` レイヤ（回帰のため活性化関数なし）に入力されます。

In [ ]:
# ワイド ネットワークとディープ ネットワークを連結
concat = Concatenate(name="concatenate")([deep, wide])

# 最終出力レイヤを定義
output = Dense(1, activation=None, name="output")(concat)

### モデルのインスタンス化とコンパイル

Functional API では、`keras.Model` を使用して、`inputs`（`Input` レイヤの辞書）と `outputs`（最終的な `Dense` レイヤ）を指定することでモデルを定義します。

In [ ]:
model = Model(inputs=list(inputs.values()), outputs=output)

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=[RootMeanSquaredError()],
)

Functional API では、`keras.utils.plot_model` がモデル構造の図を生成します。

In [ ]:
keras.utils.plot_model(model, show_layer_names=True, rankdir="LR")

## Wide & Deep モデルのトレーニング
次に、トレーニング変数を設定し、トレーニングおよび検証用のデータセットを作成して、モデルをトレーニングします。

（トレーニングを `NUM_TRAIN_EXAMPLES` と `NUM_EVALS` で表現する理由、およびこのトレーニング コードにおいてエポック数が実行する評価の回数と実質的に等しい理由の詳細については、ブログ記事 [ML Design Pattern #3: Virtual Epochs](https://medium.com/google-cloud/ml-design-pattern-3-virtual-epochs-f842296de730) を参照してください。）

In [ ]:
BATCH_SIZE = 64
NUM_TRAIN_EXAMPLES = 10000 * 10  # トレーニング データセットはリピートされ、折り返されます
NUM_EVALS = 10  # 評価を実行する回数
NUM_EVAL_EXAMPLES = 1000  # 合理的なサンプルを取得するのに十分な量

trainds = create_dataset(
    pattern="../data/taxi-train.csv", batch_size=BATCH_SIZE, mode="train"
)

evalds = create_dataset(
    pattern="../data/taxi-valid.csv", batch_size=BATCH_SIZE, mode="eval"
).take(NUM_EVAL_EXAMPLES // BATCH_SIZE)

In [ ]:
%%time
steps_per_epoch = NUM_TRAIN_EXAMPLES // (BATCH_SIZE * NUM_EVALS)

OUTDIR = "./taxi_trained"
shutil.rmtree(path=OUTDIR, ignore_errors=True)  # 毎回新しく開始

history = model.fit(
    x=trainds,
    steps_per_epoch=steps_per_epoch,
    epochs=NUM_EVALS,
    validation_data=evalds,
    callbacks=[TensorBoard(OUTDIR)],
)

前と同様に、トレーニング セットと検証セットでのトレーニングを通じて RMSE がどのように変化するか履歴を調べることができます。

In [ ]:
RMSE_COLS = ["root_mean_squared_error", "val_root_mean_squared_error"]

pd.DataFrame(history.history)[RMSE_COLS].plot()

Copyright 2025 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.